In [33]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv('/Users/hugoramos/Library/CloudStorage/OneDrive-LingaroSp.zo.o/Documents/PG/Amplified/lumen_amplified_attention(attention_all_countries).csv')
df.columns

Index(['market', 'age_group', 'gender', 'mrc_compliant_only', 'platform',
       'format', 'active_attention', 'passive_attention', 'total_attention'],
      dtype='object')

In [22]:
len(df)

2428

In [23]:
# Normalization of age_group column to 35_44
df['age_group'] = df['age_group'].str.replace('-', '_')

In [24]:
# Add device column
def add_device_column(df):
    conditions = [
        df['format'].str.contains('mobile', case=False, na=False),
        df['platform'].str.contains('mobile', case=False, na=False),
        df['platform'].str.contains('desktop', case=False, na=False),
        df['platform'].str.contains('TV', case=False, na=False)
    ]

    choices = ['Mobile', 'Mobile', 'Desktop', 'TV']

    df['device'] = np.select(conditions, choices, default=None)

    return df

 

add_device_column(df)

,market,age_group,gender,mrc_compliant_only,platform,format,active_attention,passive_attention,total_attention,device
0,US,18_24,male,False,Snapchat,AR lens,13.12,0.00,13.12,None
1,US,18_24,male,False,General Web,Outstream video,2.83,6.74,9.57,None
2,US,18_24,male,False,Facebook,Feed,2.39,3.32,5.71,None
3,US,18_24,male,False,General Web,Pre-roll video,1.77,4.15,5.93,None
4,US,18_24,male,False,Meta,Feed,1.24,0.70,1.94,None
...,...,...,...,...,...,...,...,...,...,...
2423,Germany,18+,female,True,Facebook,Feed,2.82,5.75,8.57,None
2424,Germany,18+,female,True,Pinterest,Max width video,1.35,3.87,5.22,None
2425,Germany,18+,female,True,Pinterest,Standard video,0.55,4.74,5.29,None
2426,Germany,18+,female,True,Pinterest,Standard image,0.43,4.49,4.92,None


In [25]:
df['device'].value_counts()

device
Mobile     349
TV         347
Desktop    278
Name: count, dtype: int64

In [31]:
df_with_numbers = df[df['format'].str.contains(r'\d', na=False)]
df_with_numbers[['format']].value_counts()

format                   
30 seconds                   148
15 seconds                   143
Mrec 300x250                 124
Half page 300x600             81
10 seconds                    75
Mobile leaderboard 320x50     74
Mobile banner 300x50          72
20 seconds                    66
60 seconds                    58
40 seconds                    47
Billboard 970x250             46
Non-skippable 15 seconds      40
Non-skippable 20 seconds      33
Skyscraper 160x600            26
Leaderboard 728x90            24
Non-skippable 10 seconds      17
90 seconds                    12
Non-skippable 12 seconds       2
Name: count, dtype: int64

In [26]:
# Add column attention_rate
df['video_length'] = df['format'].str.extract(r'(\d+)\s*seconds', expand=False).astype(float)

#Add active, passive, and overall attention rate: 
df['overall_attention_rate'] = df['total_attention'] / df['video_length']
df['active_attention_rate'] = df['active_attention'] / df['video_length']
df['passive_attention_rate'] = df['passive_attention'] / df['video_length']

In [27]:
df

,market,age_group,gender,mrc_compliant_only,platform,format,active_attention,passive_attention,total_attention,device,video_length,overall_attention_rate,active_attention_rate,passive_attention_rate
0,US,18_24,male,False,Snapchat,AR lens,13.12,0.00,13.12,None,NaN,NaN,NaN,NaN
1,US,18_24,male,False,General Web,Outstream video,2.83,6.74,9.57,None,NaN,NaN,NaN,NaN
2,US,18_24,male,False,Facebook,Feed,2.39,3.32,5.71,None,NaN,NaN,NaN,NaN
3,US,18_24,male,False,General Web,Pre-roll video,1.77,4.15,5.93,None,NaN,NaN,NaN,NaN
4,US,18_24,male,False,Meta,Feed,1.24,0.70,1.94,None,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2423,Germany,18+,female,True,Facebook,Feed,2.82,5.75,8.57,None,NaN,NaN,NaN,NaN
2424,Germany,18+,female,True,Pinterest,Max width video,1.35,3.87,5.22,None,NaN,NaN,NaN,NaN
2425,Germany,18+,female,True,Pinterest,Standard video,0.55,4.74,5.29,None,NaN,NaN,NaN,NaN
2426,Germany,18+,female,True,Pinterest,Standard image,0.43,4.49,4.92,None,NaN,NaN,NaN,NaN


In [28]:
df['passive_attention_rate'].value_counts()

passive_attention_rate
0.303000    7
0.233333    6
0.258000    5
0.300500    5
0.347000    5
           ..
0.429333    1
0.469667    1
0.332000    1
0.627000    1
0.258500    1
Name: count, Length: 376, dtype: int64

In [32]:
df.to_parquet('amp_new.parquet')

In [26]:
dx = pd.read_excel('amplified_media_mapping.xlsx')
dp = pd.read_parquet('amp_new.parquet')

In [27]:
dx.columns

Index(['platform', 'media_subtype', 'app_type', 'publisher'], dtype='object')

In [28]:
# Create keys using the first three columns
keys = list(zip(dx["platform"]))

# Create the separate mapping dictionaries
media_subtype_map = dict(zip(keys, dx["media_subtype"]))
app_type_map = dict(zip(keys, dx["app_type"]))
publisher_map = dict(zip(keys, dx["publisher"]))

In [29]:
publisher_map

{('Cinema',): nan,
 ('Snapchat',): 'Snapchat',
 ('Youtube',): 'YouTube',
 ('General Web',): nan,
 ('Pinterest',): 'Pinterest',
 ('Facebook',): 'Facebook',
 ('Meta',): 'Meta',
 ('Instagram',): 'Instagram',
 ('TikTok',): 'TikTok',
 ('OOH (pedestrian only)',): nan,
 ('Twitter',): 'Twitter',
 ('Spotify',): 'Spotify',
 ('General web Desktop',): nan,
 ('Twitch Desktop',): 'Twitch',
 ('BVOD Mobile',): nan,
 ('SVOD Mobile',): nan,
 ('Linear TV (free)',): nan,
 ('BVOD TV',): nan,
 ('Linear TV (paid)',): nan,
 ('SVOD TV',): nan}

In [30]:
def replace_nan_with_none(key):
    return tuple(None if pd.isna(k) else k for k in key)

media_subtype_map = {
    replace_nan_with_none(k): v
    for k, v in media_subtype_map.items()
}
app_type_map = {
    replace_nan_with_none(k): v
    for k, v in app_type_map.items()
}
publisher_map = {
    replace_nan_with_none(k): v
    for k, v in publisher_map.items()
}

In [31]:
# Create all the column by mapping each row
dp["media_subtype"] = dp.apply(
    lambda row: media_subtype_map.get((row["platform"],)),
    axis=1)

dp["app_type"] = dp.apply(
    lambda row: app_type_map.get((row["platform"],)),
    axis=1)

dp["publisher"] = dp.apply(
    lambda row: publisher_map.get((row["platform"],)),
    axis=1)

In [35]:
dp

,market,age_group,gender,mrc_compliant_only,platform,format,active_attention,passive_attention,total_attention,device,video_length,overall_attention_rate,active_attention_rate,passive_attention_rate,media_subtype,app_type,publisher
0,US,18_24,male,False,Snapchat,AR lens,13.12,0.00,13.12,None,NaN,NaN,NaN,NaN,Social,Social,Snapchat
1,US,18_24,male,False,General Web,Outstream video,2.83,6.74,9.57,None,NaN,NaN,NaN,NaN,OLV,NaN,NaN
2,US,18_24,male,False,Facebook,Feed,2.39,3.32,5.71,None,NaN,NaN,NaN,NaN,Social,Social,Facebook
3,US,18_24,male,False,General Web,Pre-roll video,1.77,4.15,5.93,None,NaN,NaN,NaN,NaN,OLV,NaN,NaN
4,US,18_24,male,False,Meta,Feed,1.24,0.70,1.94,None,NaN,NaN,NaN,NaN,Social,Social,Meta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2423,Germany,18+,female,True,Facebook,Feed,2.82,5.75,8.57,None,NaN,NaN,NaN,NaN,Social,Social,Facebook
2424,Germany,18+,female,True,Pinterest,Max width video,1.35,3.87,5.22,None,NaN,NaN,NaN,NaN,Social,Social,Pinterest
2425,Germany,18+,female,True,Pinterest,Standard video,0.55,4.74,5.29,None,NaN,NaN,NaN,NaN,Social,Social,Pinterest
2426,Germany,18+,female,True,Pinterest,Standard image,0.43,4.49,4.92,None,NaN,NaN,NaN,NaN,Social,Social,Pinterest


In [34]:
dp.to_parquet('amplify_attention.parquet')